In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("orvile/health-and-sleep-relation-2024")

print("Path to dataset files:", path)

100%|██████████████████████████████████████████████████████████████████████████████| 2.60k/2.60k [00:00<00:00, 444kB/s]

Extracting files...
Path to dataset files: C:\Users\anast\.cache\kagglehub\datasets\orvile\health-and-sleep-relation-2024\versions\1


In [1]:
import pandas as pd
import numpy as np
 
data = pd.read_csv(r'C:\Users\anast\.cache\kagglehub\datasets\orvile\health-and-sleep-relation-2024\versions\1\Health and Sleep relation 2024\Sleep_health_and_lifestyle_dataset.csv')
data

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,None
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,370,Female,59,Nurse,8.1,9,75,3,Overweight,140/95,68,7000,Sleep Apnea
370,371,Female,59,Nurse,8.0,9,75,3,Overweight,140/95,68,7000,Sleep Apnea
371,372,Female,59,Nurse,8.1,9,75,3,Overweight,140/95,68,7000,Sleep Apnea
372,373,Female,59,Nurse,8.1,9,75,3,Overweight,140/95,68,7000,Sleep Apnea


In [3]:
data['Sleep Disorder'] = np.where(data['Sleep Disorder'] != 'None', 1, 0)
data.nunique()

Person ID                  374
Gender                       2
Age                         31
Occupation                  11
Sleep Duration              27
Quality of Sleep             6
Physical Activity Level     16
Stress Level                 6
BMI Category                 4
Blood Pressure              25
Heart Rate                  19
Daily Steps                 20
Sleep Disorder               2
dtype: int64

In [4]:
data=data.drop(['Person ID','Occupation'],axis=1)
data

,Gender,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,Male,27,6.1,6,42,6,Overweight,126/83,77,4200,0
1,Male,28,6.2,6,60,8,Normal,125/80,75,10000,0
2,Male,28,6.2,6,60,8,Normal,125/80,75,10000,0
3,Male,28,5.9,4,30,8,Obese,140/90,85,3000,1
4,Male,28,5.9,4,30,8,Obese,140/90,85,3000,1
...,...,...,...,...,...,...,...,...,...,...,...
369,Female,59,8.1,9,75,3,Overweight,140/95,68,7000,1
370,Female,59,8.0,9,75,3,Overweight,140/95,68,7000,1
371,Female,59,8.1,9,75,3,Overweight,140/95,68,7000,1
372,Female,59,8.1,9,75,3,Overweight,140/95,68,7000,1


In [5]:
data=data.drop(['Blood Pressure'],axis=1)
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

def encode_g(df, column_name):
    categories = ['Male','Female']

    encoder = OrdinalEncoder(categories=[categories], handle_unknown='use_encoded_value', unknown_value=-1)

    df[column_name] = encoder.fit_transform(df[[column_name]])

    return df

df = pd.DataFrame(data)

print("Original DataFrame:\n", df)

df = encode_g(df, 'Gender')

print("\nEncoded DataFrame:\n", df)

Original DataFrame:
      Gender  Age  Sleep Duration  Quality of Sleep  Physical Activity Level  \
0      Male   27             6.1                 6                       42   
1      Male   28             6.2                 6                       60   
2      Male   28             6.2                 6                       60   
3      Male   28             5.9                 4                       30   
4      Male   28             5.9                 4                       30   
..      ...  ...             ...               ...                      ...   
369  Female   59             8.1                 9                       75   
370  Female   59             8.0                 9                       75   
371  Female   59             8.1                 9                       75   
372  Female   59             8.1                 9                       75   
373  Female   59             8.1                 9                       75   

     Stress Level BMI Category

In [9]:
def encode_bmi(df, column_name):
    categories = ['Normal','Normal Weight','Overweight','Obese']

    encoder = OrdinalEncoder(categories=[categories], handle_unknown='use_encoded_value', unknown_value=-1)

    df[column_name] = encoder.fit_transform(df[[column_name]])

    return df

df = pd.DataFrame(data)

print("Original DataFrame:\n", df)

df = encode_bmi(df, 'BMI Category')

print("\nEncoded DataFrame:\n", df)

Original DataFrame:
      Gender  Age  Sleep Duration  Quality of Sleep  Physical Activity Level  \
0       0.0   27             6.1                 6                       42   
1       0.0   28             6.2                 6                       60   
2       0.0   28             6.2                 6                       60   
3       0.0   28             5.9                 4                       30   
4       0.0   28             5.9                 4                       30   
..      ...  ...             ...               ...                      ...   
369     1.0   59             8.1                 9                       75   
370     1.0   59             8.0                 9                       75   
371     1.0   59             8.1                 9                       75   
372     1.0   59             8.1                 9                       75   
373     1.0   59             8.1                 9                       75   

     Stress Level BMI Category

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder  
from sklearn.preprocessing import StandardScaler  

def perform_svm_classification(df, target_column, feature_columns, kernel='linear', C=2.0, gamma='scale', test_size=0.7, random_state=42):
    X = data[feature_columns]
    y = data[target_column]  
    X = pd.get_dummies(X, columns=['Gender', 'BMI Category'], drop_first=True)  
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.7, random_state=42)

    print(f"Shape of y_train: {y_train.shape}") 
    print(f"Shape of y_test: {y_test.shape}")   


    model = SVC(kernel=kernel, random_state=42)
    model.fit(X_train, y_train) 
    y_pred = model.predict(X_test)

    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

    metrics = classification_report(y_test, y_pred, output_dict=True)

    coefficients_dict = {}
    if kernel == 'linear':
        try:
            coefficients_dict = dict(zip(X.columns, model.coef_[0])) 
        except AttributeError:
            print("Coefficients not available for non-linear kernel.")

    return model, predictions_df, metrics, coefficients_dict

In [22]:
target_column = 'Sleep Disorder'
feature_columns = ['Gender','Age','Sleep Duration','Quality of Sleep','Physical Activity Level','Stress Level','BMI Category','Heart Rate','Daily Steps']

model, predictions_df, metrics, coefficients_dict = perform_svm_classification(data, target_column, feature_columns, kernel='linear')

print("SVM Model:", model)
print("\nPredictions:")
print(predictions_df)
print("\nEvaluation Metrics:")
print(metrics)
print("\nCoefficients (for linear kernel):")
print(coefficients_dict)

Shape of y_train: (112,)
Shape of y_test: (262,)
SVM Model: SVC(kernel='linear', random_state=42)

Predictions:
     Actual  Predicted
329       0          0
33        0          0
15        0          0
325       0          0
57        0          0
..      ...        ...
254       1          0
356       1          1
4         1          1
256       1          0
333       0          0

[262 rows x 2 columns]

Evaluation Metrics:
{'0': {'precision': 0.8366013071895425, 'recall': 0.8590604026845637, 'f1-score': 0.847682119205298, 'support': 149}, '1': {'precision': 0.8073394495412844, 'recall': 0.7787610619469026, 'f1-score': 0.7927927927927928, 'support': 113}, 'accuracy': 0.8244274809160306, 'macro avg': {'precision': 0.8219703783654135, 'recall': 0.8189107323157332, 'f1-score': 0.8202374559990454, 'support': 262}, 'weighted avg': {'precision': 0.8239807349977365, 'recall': 0.8244274809160306, 'f1-score': 0.824008478424332, 'support': 262}}

Coefficients (for linear kernel):
{'Age': 0.

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression  # Changed from SVC
from sklearn.metrics import mean_squared_error, r2_score  # Changed from classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

def perform_linear_regression(df, target_column, feature_columns, test_size=0.7, random_state=42):
   
    X = data[feature_columns] 
    y = data[target_column]  
    X = pd.get_dummies(X, columns=['Gender', 'BMI Category'], drop_first=True)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = LinearRegression()  
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    predictions_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    metrics = {'Mean Squared Error': mse, 'R-squared': r2}

    coefficients_dict = dict(zip(X.columns, model.coef_))

    return model, predictions_df, metrics, coefficients_dict

In [24]:
target_column = 'Quality of Sleep'
feature_columns = ['Gender','Age','Sleep Duration','Sleep Disorder','Physical Activity Level','Stress Level','BMI Category','Heart Rate','Daily Steps']

model, predictions_df, metrics, coefficients_dict = perform_linear_regression(data, target_column, feature_columns)

print("Linear Model:", model)
print("\nPredictions:")
print(predictions_df)
print("\nEvaluation Metrics:")
print(metrics)
print("\nCoefficients:")
print(coefficients_dict)

Linear Model: LinearRegression()

Predictions:
     Actual  Predicted
329       9   9.258026
33        6   5.866736
15        6   5.627736
325       9   9.258026
57        6   5.901153
..      ...        ...
254       7   7.194873
356       9   8.724888
4         4   4.827124
256       7   7.215121
333       9   9.292443

[262 rows x 2 columns]

Evaluation Metrics:
{'Mean Squared Error': 0.11422585698349866, 'R-squared': 0.9218354577494913}

Coefficients:
{'Age': 0.4753018294767745, 'Sleep Duration': 0.16056742705060398, 'Sleep Disorder': -0.06018980216273159, 'Physical Activity Level': 0.13915744781932485, 'Stress Level': -0.6192766195060859, 'Heart Rate': 0.0018400382050223242, 'Daily Steps': -0.05647115462894328, 'Gender_1.0': -0.03108851057627772, 'BMI Category_1.0': 0.06701412048159752, 'BMI Category_2.0': -0.3915605063745741, 'BMI Category_3.0': -0.10452807627181766}


In [25]:
target_column = 'Sleep Duration'
feature_columns = ['Gender','Age','Quality of Sleep','Sleep Disorder','Physical Activity Level','Stress Level','BMI Category','Heart Rate','Daily Steps']

model, predictions_df, metrics, coefficients_dict = perform_linear_regression(data, target_column, feature_columns)

print("Linear Model:", model)
print("\nPredictions:")
print(predictions_df)
print("\nEvaluation Metrics:")
print(metrics)
print("\nCoefficients:")
print(coefficients_dict)

Linear Model: LinearRegression()

Predictions:
     Actual  Predicted
329     8.5   8.122994
33      6.1   6.318057
15      6.0   6.070544
325     8.5   8.122994
57      6.0   6.337337
..      ...        ...
254     6.5   6.787319
356     8.0   8.052601
4       5.9   5.113109
256     6.6   6.787319
333     8.4   8.142275

[262 rows x 2 columns]

Evaluation Metrics:
{'Mean Squared Error': 0.10433324767672619, 'R-squared': 0.8337257771131347}

Coefficients:
{'Age': 0.16764035579843586, 'Quality of Sleep': 0.3128612883433263, 'Sleep Disorder': 0.06168159222074188, 'Physical Activity Level': 0.12162598022702129, 'Stress Level': -0.38973795128179856, 'Heart Rate': 0.15315894289866194, 'Daily Steps': -0.06755686320248695, 'Gender_1.0': -0.13582203154669892, 'BMI Category_1.0': -0.01596192860869343, 'BMI Category_2.0': -0.28421907989799045, 'BMI Category_3.0': -0.17628662975086828}


In [26]:
data.corr()

,Gender,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder
Gender,1.000000,0.596358,0.121579,0.291366,-0.001454,-0.396018,0.303881,-0.217105,0.014509,0.285824
Age,0.596358,1.000000,0.344709,0.473734,0.178993,-0.422344,0.453232,-0.225606,0.057973,0.432007
Sleep Duration,0.121579,0.344709,1.000000,0.883213,0.212360,-0.811023,-0.360153,-0.516455,-0.039533,-0.338622
Quality of Sleep,0.291366,0.473734,0.883213,1.000000,0.192896,-0.898752,-0.326983,-0.659865,0.016791,-0.310984
Physical Activity Level,-0.001454,0.178993,0.212360,0.192896,1.000000,-0.034134,0.065020,0.136971,0.772723,0.069787
Stress Level,-0.396018,-0.422344,-0.811023,-0.898752,-0.034134,1.000000,0.160531,0.670026,0.186829,0.181685
BMI Category,0.303881,0.453232,-0.360153,-0.326983,0.065020,0.160531,1.000000,0.435153,-0.100081,0.796847
Heart Rate,-0.217105,-0.225606,-0.516455,-0.659865,0.136971,0.670026,0.435153,1.000000,-0.030309,0.330254
Daily Steps,0.014509,0.057973,-0.039533,0.016791,0.772723,0.186829,-0.100081,-0.030309,1.000000,-0.026575
Sleep Disorder,0.285824,0.432007,-0.338622,-0.310984,0.069787,0.181685,0.796847,0.330254,-0.026575,1.000000
